# Neuron Architecture Demo

本notebook展示新型神经元架构的核心功能。

## 内容
1. SCH-Neuron: 脉冲率自适应
2. HM-Neuron: 遗忘曲线对比
3. NG-Neuron: 神经调质门控
4. ESB-Neuron: 符号识别

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')

from neuron_arch import (
    AdaptiveThresholdSCH,
    PatternSeparationHM,
    NormalizedNG,
    ESBNeuronLayer
)

np.random.seed(42)

## 1. SCH-Neuron: 脉冲率自适应

展示自适应阈值如何维持目标脉冲率。

In [ ]:
# 创建SCH神经元
sch = AdaptiveThresholdSCH(n_neurons=64, target_spike_rate=0.1)

# 记录脉冲率变化
spike_rates = []
thresholds = []

for step in range(200):
    input_current = np.random.randn(64) * 0.5
    spikes, _ = sch.step(input_current)
    
    spike_rates.append(np.mean(spikes))
    thresholds.append(np.mean(sch.v_th))

# 绘图
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(spike_rates, label='实际脉冲率')
axes[0].axhline(y=0.1, color='r', linestyle='--', label='目标脉冲率')
axes[0].set_xlabel('步数')
axes[0].set_ylabel('脉冲率')
axes[0].set_title('SCH脉冲率自适应')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds)
axes[1].set_xlabel('步数')
axes[1].set_ylabel('平均阈值')
axes[1].set_title('自适应阈值变化')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sch_spike_rate.png', dpi=100)
plt.show()

print(f'最终脉冲率: {np.mean(spike_rates[-50:]):.4f}')
print(f'目标脉冲率: 0.1000')

## 2. HM-Neuron: 遗忘曲线对比

对比有无HM层的遗忘曲线。

In [ ]:
# 生成干扰任务
def generate_task(task_id, n=100):
    x = np.linspace(0, 1, n)
    np.random.seed(42 + task_id)
    noise = np.random.randn(n) * 0.03
    
    if task_id == 0:
        y = np.sin(2*np.pi*x)
    elif task_id == 1:
        y = 2*x - 1
    elif task_id == 2:
        y = np.exp(x) - 1.5
    else:
        y = np.abs(2*x - 1) - 0.5
    
    return x, y + noise

tasks = [generate_task(i) for i in range(4)]

In [ ]:
# 有HM的系统
hm_with = PatternSeparationHM(16, 8, hippocampus_lr=0.05, cortex_lr=0.01)
forgetting_with_hm = []

for task_id in range(4):
    x_data, y_data = tasks[task_id]
    
    for x, y in zip(x_data, y_data):
        inp = np.zeros(16)
        inp[0] = x
        target = np.zeros(8)
        target[0] = y
        hm_with.learn(inp, target, task_id=task_id)
    
    hm_with.consolidate(n_replay=20)
    
    x0, y0 = tasks[0]
    total_error = 0
    for x, y in zip(x0[:20], y0[:20]):
        inp = np.zeros(16)
        inp[0] = x
        pred = hm_with.forward(inp)
        total_error += (y - pred[0])**2
    forgetting_with_hm.append(total_error / 20)

# 无HM的系统（简单网络）
class SimpleNet:
    def __init__(self):
        self.W = np.random.randn(8, 16) * 0.1
    
    def forward(self, x):
        return self.W @ x
    
    def learn(self, x, target, lr=0.05):
        pred = self.forward(x)
        error = target - pred
        self.W += lr * np.outer(error, x)

simple_net = SimpleNet()
forgetting_no_hm = []

for task_id in range(4):
    x_data, y_data = tasks[task_id]
    
    for x, y in zip(x_data, y_data):
        inp = np.zeros(16)
        inp[0] = x
        target = np.zeros(8)
        target[0] = y
        simple_net.learn(inp, target)
    
    x0, y0 = tasks[0]
    total_error = 0
    for x, y in zip(x0[:20], y0[:20]):
        inp = np.zeros(16)
        inp[0] = x
        pred = simple_net.forward(inp)
        total_error += (y - pred[0])**2
    forgetting_no_hm.append(total_error / 20)

In [ ]:
# 绘制遗忘曲线
plt.figure(figsize=(10, 6))

plt.plot(range(1, 5), forgetting_with_hm, 'b-o', linewidth=2, markersize=8, label='有HM层')
plt.plot(range(1, 5), forgetting_no_hm, 'r-s', linewidth=2, markersize=8, label='无HM层')

plt.xlabel('任务阶段', fontsize=12)
plt.ylabel('任务0 MSE', fontsize=12)
plt.title('遗忘曲线对比: HM-Neuron vs 简单网络', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, 5), ['任务0后', '任务1后', '任务2后', '任务3后'])

# 标注最差遗忘
worst_with_hm = max(forgetting_with_hm)
worst_no_hm = max(forgetting_no_hm)

plt.axhline(y=worst_with_hm, color='b', linestyle=':', alpha=0.5)
plt.axhline(y=worst_no_hm, color='r', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('forgetting_curve.png', dpi=100)
plt.show()

print(f'有HM最差MSE: {worst_with_hm:.4f}')
print(f'无HM最差MSE: {worst_no_hm:.4f}')
print(f'HM缓解效果: {(worst_no_hm - worst_with_hm) / worst_no_hm * 100:.1f}%')

## 3. NG-Neuron: 神经调质门控

展示DA如何控制学习率。

In [ ]:
ng = NormalizedNG(base_lr=0.01)

# 不同奖励水平
rewards = [1.0, 0.5, 0.0, 0.8, 0.3, 1.0, 0.0]
da_history = []
lr_history = []

for r in rewards:
    ng.update_from_reward(r)
    da_history.append(ng.da)
    lr_history.append(ng.compute_effective_lr())

# 绘图
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(len(rewards)), rewards, color='green', alpha=0.7, label='奖励')
axes[0].plot(range(len(da_history)), da_history, 'r-o', label='DA水平')
axes[0].set_xlabel('步数')
axes[0].set_ylabel('值')
axes[0].set_title('奖励与DA响应')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(lr_history, 'b-o')
axes[1].set_xlabel('步数')
axes[1].set_ylabel('有效学习率')
axes[1].set_title('NG门控学习率')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ng_gating.png', dpi=100)
plt.show()

## 4. ESB-Neuron: 符号识别

展示正交模式如何实现符号区分。

In [ ]:
esb = ESBNeuronLayer(16, 8, 4)

# 正交模式
patterns = np.zeros((4, 16))
for i in range(4):
    patterns[i, i*4:(i+1)*4] = 1.0
    patterns[i] += np.random.randn(16) * 0.05
    patterns[i] /= np.linalg.norm(patterns[i])

# 训练
for epoch in range(100):
    for symbol_id in range(4):
        esb.learn(patterns[symbol_id], symbol_id, lr=0.1)

# 测试
confidences = []
for symbol_id in range(4):
    latent = esb.encode_embodied(patterns[symbol_id])
    pred_id, probs, conf = esb.decode_symbol(latent)
    confidences.append(conf)
    print(f'符号{symbol_id}: 预测={pred_id}, 置信度={conf:.3f}')

# 绘图
plt.figure(figsize=(8, 5))
plt.bar(range(4), confidences, color='steelblue')
plt.axhline(y=0.25, color='r', linestyle='--', label='随机基线')
plt.xlabel('符号ID')
plt.ylabel('置信度')
plt.title('ESB符号识别置信度')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('esb_symbols.png', dpi=100)
plt.show()

print(f'\n平均置信度: {np.mean(confidences):.3f} (随机=0.25)')

## 总结

本demo展示了四种新型神经元架构的核心功能：

| 架构 | 功能 | 验证 |
|------|------|------|
| SCH-Neuron | 脉冲率自适应 | ✓ 维持目标率 |
| HM-Neuron | 遗忘缓解 | ✓ 曲线对比 |
| NG-Neuron | 学习率门控 | ✓ DA响应 |
| ESB-Neuron | 符号识别 | ✓ 高置信度 |